In [1]:
import pandas as pd
import urllib.request
import zipfile
import os

# Download the official UCI zip
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"
urllib.request.urlretrieve(url, "bank.zip")

# Extract bank-full.csv
with zipfile.ZipFile("bank.zip", "r") as zip_ref:
    zip_ref.extractall(".")

# Load the dataset (note the semicolon separator)
df = pd.read_csv("bank-full.csv", sep=";")

print("Shape of the dataset:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names and data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution (y):")
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True))

/home/cloud/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
/home/cloud/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Shape of the dataset: (45211, 17)

First 5 rows:
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  

Column names and data types:
age          int64
job          

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Target encoding
df["y"] = df["y"].map({"no": 0, "yes": 1})

# Features and target
X = df.drop("y", axis=1)
y = df["y"]

# Column groups
numerical_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
categorical_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), categorical_cols)
    ]
)

# Fit and transform
X_processed = preprocessor.fit_transform(X)

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Positive class percentage in train:", round(y_train.mean(), 4))
print("Positive class percentage in test:", round(y_test.mean(), 4))

Training set shape: (36168, 42)
Test set shape: (9043, 42)
Positive class percentage in train: 0.117
Positive class percentage in test: 0.117


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# ========== 1. Load & Preprocess ==========
df = pd.read_csv("bank-full.csv", sep=";")
df["y"] = df["y"].map({"no": 0, "yes": 1})

X = df.drop("y", axis=1)
y = df["y"]

numerical_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
categorical_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), categorical_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

# ========== 2. Train Models & Calculate Metrics ==========
results = []

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
y_proba = lr.predict_proba(X_test)[:, 1]
results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
})

# 2. Decision Tree
dt = DecisionTreeClassifier(random_state=42, max_depth=10, class_weight="balanced")
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)
y_proba = dt.predict_proba(X_test)[:, 1]
results.append({
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
})

# 3. K-Nearest Neighbors
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
y_proba = knn.predict_proba(X_test)[:, 1]
results.append({
    "Model": "KNN",
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
})

# 4. Naive Bayes (Gaussian)
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)
y_proba = nb.predict_proba(X_test)[:, 1]
results.append({
    "Model": "Naive Bayes",
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
})

# 5. Support Vector Machine (faster version)
base_svm = SVC(kernel="linear", random_state=42, class_weight="balanced")
svm = CalibratedClassifierCV(base_svm, cv=3)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
y_proba = svm.predict_proba(X_test)[:, 1]
results.append({
    "Model": "SVM",
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
})

# ========== 3. Show Results ==========
results_df = pd.DataFrame(results)
results_df = results_df.round(4)
print("\n========== Comparison Table ==========")
print(results_df.to_string(index=False))

Training set shape: (36168, 42)
Test set shape: (9043, 42)

========== Comparison Table ==========
              Model  Accuracy    AUC  Precision  Recall     F1    MCC
Logistic Regression    0.8458 0.9079     0.4184  0.8147 0.5529 0.5094
      Decision Tree    0.8262 0.8581     0.3865  0.8270 0.5268 0.4845
                KNN    0.8965 0.8368     0.5962  0.3573 0.4468 0.4094
        Naive Bayes    0.8639 0.8088     0.4282  0.4877 0.4560 0.3797
                SVM    0.8999 0.9077     0.6393  0.3318 0.4368 0.4132


In [8]:
import joblib
import os

# Create folders
os.makedirs("model", exist_ok=True)

# Save the five trained models
joblib.dump(lr, "model/logistic_regression.joblib")
joblib.dump(dt, "model/decision_tree.joblib")
joblib.dump(knn, "model/knn.joblib")
joblib.dump(nb, "model/naive_bayes.joblib")
joblib.dump(svm, "model/svm.joblib")

# Save the preprocessor (important for the app)
joblib.dump(preprocessor, "model/preprocessor.joblib")

# Save the test set as CSV (for the Streamlit upload feature)
# We need original features + target, not the processed version
# Re-create a clean test CSV from the original data
_, X_test_raw, _, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
test_df = X_test_raw.copy()
test_df["y"] = y_test_raw.values
test_df.to_csv("test_data.csv", index=False)

print("Models and test_data.csv saved successfully!")
print("Files created:")
print("- model/logistic_regression.joblib")
print("- model/decision_tree.joblib")
print("- model/knn.joblib")
print("- model/naive_bayes.joblib")
print("- model/svm.joblib")
print("- model/preprocessor.joblib")
print("- test_data.csv")

Models and test_data.csv saved successfully!
Files created:
- model/logistic_regression.joblib
- model/decision_tree.joblib
- model/knn.joblib
- model/naive_bayes.joblib
- model/svm.joblib
- model/preprocessor.joblib
- test_data.csv
